1、prompt构建提示词：
    PromptTemplate:对单条提示词进行格式化,
    ChatPromptTemplate:对多条提示词进行格式化
    RichPromptTemplate:动态构建提示词

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from langchain_openai.chat_models import ChatOpenAI
from Config.load_key import open_key
from Config.load_key import base_model
from Config.load_key import base_url
from llama_index.embeddings.dashscope import DashScopeEmbedding, DashScopeTextEmbeddingModels, \
    DashScopeTextEmbeddingType

#初始化通义千问的embedding模型
Settings.embed_model = DashScopeEmbedding(
    model_name=DashScopeTextEmbeddingModels.TEXT_EMBEDDING_V2,
    text_type=DashScopeTextEmbeddingType.TEXT_TYPE_DOCUMENT,
    api_key=open_key
)

Settings.llm = ChatOpenAI(
    model=base_model,
    base_url=base_url,
    api_key=open_key
)

#读取文档
document = SimpleDirectoryReader('Source').load_data()
#创建向量索引
index = VectorStoreIndex.from_documents(document)

#创建查询引擎入口
query_engine = index.as_query_engine()
resp = query_engine.query('身高170，体重55kg，应该选什么衣服在南方冬天的时候，尽量穿的舒服一点')
resp


In [ ]:
from llama_index.core import PromptTemplate

template = ("把语句 \"{text}  \"翻译成{language}")
#构建提示词模版
prompt_template = PromptTemplate(template=template)
#将提示词模版格式化提具体提示词
prompt = prompt_template.format(text="hello world", language="python")
print(prompt)

#将提示词模版格式化为聊天信息
messages = prompt_template.format_messages(text="hello world", language="python")
print(messages)

In [ ]:
from llama_index.core import ChatPromptTemplate
from llama_index.core.llms import ChatMessage, MessageRole

message_templates = [
    ChatMessage(content="你是一个专业的翻译，负责把用户语言翻译成{language}", role=MessageRole.SYSTEM),
    ChatMessage(content="{text}", role=MessageRole.USER)
]
#构建模版
chat_template = ChatPromptTemplate(message_templates=message_templates)
#格式化文本
prompt = chat_template.format(text="hello world", language="python")
print(prompt)
print('===============================')
#格式化消息
messages = chat_template.format_messages(text="hello world", language="python")
print(messages)


In [ ]:
from llama_index.core.prompts import RichPromptTemplate

prompt_template = RichPromptTemplate(
    "把用户的问题：{{text}}翻译成  {{language}}"
)
prompt = prompt_template.format(text="hello world", language="python")
print(prompt)

#####llamaindex中使用了大量的提示词，可以用get_prompt方法获取当前组件里的提示词，并且用户可以修改

In [ ]:
prompt_dict = query_engine.get_prompts()
for key in prompt_dict:
    print(key + '>>>>')
    print(prompt_dict[key])
    print("--------===")

In [ ]:
query_template_str = (
    "以下是参考信息：\n"
    "============\n"
    "{{context_str}}\n"
    "============\n"
    "根据用户参考回答用户的问题，带上于谦的语气，幽默一点\n"
    "问题{{query_str}}\n"
    "答案："
)

qa_prompt_str = RichPromptTemplate(query_template_str)
##更新模版
query_engine.update_prompts({
    "response_synthesizer:text_qa_template": qa_prompt_str,
})

resp = query_engine.query('身高170，体重55kg，应该选什么衣服在南方冬天的时候，尽量穿的舒服一点')
resp